In [1]:
from pathlib import Path
import pandas as pd


def ler_csv_anp(arquivo, **kwargs):
    try:
        return pd.read_csv(arquivo, encoding="utf-8-sig", **kwargs)
    except UnicodeDecodeError:
        return pd.read_csv(arquivo, encoding="cp1252", **kwargs)


In [2]:
# Localiza os arquivos brutos a partir da raiz do projeto, independentemente do CWD do kernel.

RAIZ = Path.cwd()
if not (RAIZ / "Dados").exists() and (RAIZ.parent / "Dados").exists():
    RAIZ = RAIZ.parent
assert (RAIZ / "Dados/Bruto").exists(), "Abra o projeto ou confira a pasta Dados/Bruto."

PASTA_BRUTOS = RAIZ / "Dados/Bruto"
print("Raiz do projeto:", RAIZ)
print("Pasta dos dados brutos:", PASTA_BRUTOS)

Raiz do projeto: c:\Users\super\Desktop\Leo\faculdade\Tcc analise combustivel\projeto\TCC_Combustiveis_revisado_20261014
Pasta dos dados brutos: c:\Users\super\Desktop\Leo\faculdade\Tcc analise combustivel\projeto\TCC_Combustiveis_revisado_20261014\Dados\Bruto


In [3]:
# Localiza todos os arquivos CSV existentes na pasta de dados brutos

arquivos_csv = sorted(PASTA_BRUTOS.glob("*.csv"))

print(f"Arquivos encontrados: {len(arquivos_csv)}")

for arquivo in arquivos_csv:
    print(arquivo.name)

Arquivos encontrados: 31
ca-2011-01.csv
ca-2011-02.csv
ca-2012-01.csv
ca-2012-02.csv
ca-2013-01.csv
ca-2013-02.csv
ca-2014-01.csv
ca-2014-02.csv
ca-2015-01.csv
ca-2015-02.csv
ca-2016-01.csv
ca-2016-02.csv
ca-2017-01.csv
ca-2017-02.csv
ca-2018-01.csv
ca-2018-02.csv
ca-2019-01.csv
ca-2019-02.csv
ca-2020-01.csv
ca-2020-02.csv
ca-2021-01.csv
ca-2021-02.csv
ca-2022-01.csv
ca-2022-02.csv
ca-2023-01.csv
ca-2023-02.csv
ca-2024-01.csv
ca-2024-02.csv
ca-2025-01.csv
ca-2025-02.csv
ca-2026-01.csv


In [4]:
# Lê o primeiro arquivo CSV da lista para conferir as colunas e a estrutura dos dados

arquivo_teste = arquivos_csv[0]

df_teste = ler_csv_anp(
    arquivo_teste,
    sep=";"
)

print("Arquivo utilizado no teste:")
print(arquivo_teste.name)

print("\nQuantidade de registros:")
print(len(df_teste))

print("\nQuantidade de colunas:")
print(len(df_teste.columns))

print("\nColunas encontradas:")
print(df_teste.columns.tolist())

Arquivo utilizado no teste:
ca-2011-01.csv

Quantidade de registros:
649693

Quantidade de colunas:
16

Colunas encontradas:
['Regiao - Sigla', 'Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Valor de Compra', 'Unidade de Medida', 'Bandeira']


In [5]:
# Confere se todos os arquivos CSV possuem as mesmas colunas do arquivo de teste

colunas_referencia = df_teste.columns.tolist()
arquivos_com_diferenca = []

for arquivo in arquivos_csv:
    df_temp = ler_csv_anp(
        arquivo,
        sep=";",
        nrows=5
    )
    colunas_arquivo = df_temp.columns.tolist()
    if colunas_arquivo != colunas_referencia:
        arquivos_com_diferenca.append(arquivo.name)

print("Arquivos verificados:", len(arquivos_csv))
print("Arquivos com estrutura diferente:", len(arquivos_com_diferenca))

if arquivos_com_diferenca:
    print("\nArquivos com diferença:")
    for arquivo in arquivos_com_diferenca:
        print(arquivo)
else:
    print("\nTodos os arquivos possuem a mesma estrutura de colunas.")

Arquivos verificados: 31
Arquivos com estrutura diferente: 0

Todos os arquivos possuem a mesma estrutura de colunas.


In [6]:
# Define a lista oficial de municípios que serão considerados no recorte da Região Metropolitana de João Pessoa

MUNICIPIOS_RMJP = [
    "JOAO PESSOA",
    "SANTA RITA",
    "BAYEUX",
    "CABEDELO",
    "CONDE",
    "LUCENA",
    "ALHANDRA",
    "CAAPORA",
    "PITIMBU",
    "PEDRAS DE FOGO",
    "CRUZ DO ESPIRITO SANTO",
    "RIO TINTO"
]

print("Quantidade de municípios definidos:", len(MUNICIPIOS_RMJP))

for municipio in MUNICIPIOS_RMJP:
    print(municipio)

Quantidade de municípios definidos: 12
JOAO PESSOA
SANTA RITA
BAYEUX
CABEDELO
CONDE
LUCENA
ALHANDRA
CAAPORA
PITIMBU
PEDRAS DE FOGO
CRUZ DO ESPIRITO SANTO
RIO TINTO


In [7]:
# Verifica quais municípios da RMJP aparecem nos arquivos brutos da ANP

municipios_encontrados = set()

for arquivo in arquivos_csv:
    df_temp = ler_csv_anp(
        arquivo,
        sep=";",
        usecols=["Estado - Sigla", "Municipio"]
    )
    df_temp = df_temp[df_temp["Estado - Sigla"] == "PB"]
    municipios_arquivo = set(
        df_temp["Municipio"].astype(str).str.strip().str.upper().unique()
    )
    municipios_encontrados.update(municipios_arquivo)

municipios_rmjp_encontrados = [
    municipio for municipio in MUNICIPIOS_RMJP if municipio in municipios_encontrados
]
municipios_rmjp_ausentes = [
    municipio for municipio in MUNICIPIOS_RMJP if municipio not in municipios_encontrados
]

print("Municípios da RMJP encontrados na base:")
for municipio in municipios_rmjp_encontrados:
    print(municipio)
print("\nQuantidade encontrada:", len(municipios_rmjp_encontrados))
print("\nMunicípios da RMJP ausentes na base:")
if municipios_rmjp_ausentes:
    for municipio in municipios_rmjp_ausentes:
        print(municipio)
else:
    print("Nenhum")

C:\Users\super\AppData\Local\Temp\ipykernel_6688\3696483139.py:7: DtypeWarning: Columns (1: Estado - Sigla, 2: Municipio) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(arquivo, encoding="utf-8-sig", **kwargs)


Municípios da RMJP encontrados na base:
JOAO PESSOA
SANTA RITA
BAYEUX
CABEDELO

Quantidade encontrada: 4

Municípios da RMJP ausentes na base:
CONDE
LUCENA
ALHANDRA
CAAPORA
PITIMBU
PEDRAS DE FOGO
CRUZ DO ESPIRITO SANTO
RIO TINTO


In [8]:
# Verifica variações de escrita dos municípios da RMJP na base da ANP
# removendo acentos, espaços extras e diferenças entre maiúsculas/minúsculas

import unicodedata

def normalizar_texto(texto):
    texto = str(texto).strip().upper()
    texto = unicodedata.normalize("NFKD", texto)
    texto = "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )
    return " ".join(texto.split())

municipios_pb = set()
for arquivo in arquivos_csv:
    df_temp = ler_csv_anp(
        arquivo,
        sep=";",
        usecols=["Estado - Sigla", "Municipio"]
    )
    df_temp = df_temp[df_temp["Estado - Sigla"] == "PB"]
    municipios_pb.update(
        df_temp["Municipio"].dropna().astype(str).unique()
    )

municipios_pb_normalizados = {
    normalizar_texto(municipio): municipio for municipio in municipios_pb
}
print("Verificação dos municípios da RMJP:\n")
for municipio in MUNICIPIOS_RMJP:
    municipio_normalizado = normalizar_texto(municipio)
    if municipio_normalizado in municipios_pb_normalizados:
        print(f"ENCONTRADO: {municipio} -> {municipios_pb_normalizados[municipio_normalizado]}")
    else:
        print(f"NÃO ENCONTRADO: {municipio}")

C:\Users\super\AppData\Local\Temp\ipykernel_6688\3696483139.py:7: DtypeWarning: Columns (1: Estado - Sigla, 2: Municipio) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(arquivo, encoding="utf-8-sig", **kwargs)


Verificação dos municípios da RMJP:

ENCONTRADO: JOAO PESSOA -> JOAO PESSOA
ENCONTRADO: SANTA RITA -> SANTA RITA
ENCONTRADO: BAYEUX -> BAYEUX
ENCONTRADO: CABEDELO -> CABEDELO
NÃO ENCONTRADO: CONDE
NÃO ENCONTRADO: LUCENA
NÃO ENCONTRADO: ALHANDRA
NÃO ENCONTRADO: CAAPORA
NÃO ENCONTRADO: PITIMBU
NÃO ENCONTRADO: PEDRAS DE FOGO
NÃO ENCONTRADO: CRUZ DO ESPIRITO SANTO
NÃO ENCONTRADO: RIO TINTO


In [9]:
# Define os municípios que realmente possuem dados disponíveis na base da ANP
# e que serão utilizados no recorte final do estudo

MUNICIPIOS_ESTUDO = [
    "JOAO PESSOA",
    "SANTA RITA",
    "BAYEUX",
    "CABEDELO"
]

print("Municípios utilizados no estudo:")

for municipio in MUNICIPIOS_ESTUDO:
    print(municipio)

Municípios utilizados no estudo:
JOAO PESSOA
SANTA RITA
BAYEUX
CABEDELO


In [10]:
# Verifica quantos registros de GASOLINA e ETANOL existem
# em cada município do estudo ao longo dos anos

resumo_cobertura = []
for arquivo in arquivos_csv:
    df_temp = ler_csv_anp(
        arquivo,
        sep=";",
        usecols=["Estado - Sigla", "Municipio", "Produto", "Data da Coleta"]
    )
    df_temp = df_temp[df_temp["Estado - Sigla"] == "PB"].copy()
    df_temp["Municipio"] = df_temp["Municipio"].astype(str).str.strip().str.upper()
    df_temp["Produto"] = df_temp["Produto"].astype(str).str.strip().str.upper()
    df_temp = df_temp[
        df_temp["Municipio"].isin(MUNICIPIOS_ESTUDO)
        & df_temp["Produto"].isin(["GASOLINA", "ETANOL"])
    ]
    df_temp["Data da Coleta"] = pd.to_datetime(
        df_temp["Data da Coleta"], format="%d/%m/%Y", errors="coerce"
    )
    df_temp["Ano"] = df_temp["Data da Coleta"].dt.year
    resumo_cobertura.append(
        df_temp.groupby(["Municipio", "Produto", "Ano"]).size().reset_index(name="Registros")
    )

cobertura = pd.concat(resumo_cobertura, ignore_index=True)
cobertura = cobertura.groupby(
    ["Municipio", "Produto", "Ano"], as_index=False
)["Registros"].sum()
print(cobertura.to_string(index=False))

C:\Users\super\AppData\Local\Temp\ipykernel_6688\3696483139.py:7: DtypeWarning: Columns (1: Estado - Sigla, 2: Municipio, 10: Produto, 11: Data da Coleta) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(arquivo, encoding="utf-8-sig", **kwargs)


  Municipio  Produto  Ano  Registros
     BAYEUX   ETANOL 2011        432
     BAYEUX   ETANOL 2012        415
     BAYEUX   ETANOL 2013        404
     BAYEUX   ETANOL 2014        366
     BAYEUX   ETANOL 2015        323
     BAYEUX   ETANOL 2016        310
     BAYEUX   ETANOL 2017        243
     BAYEUX   ETANOL 2018        316
     BAYEUX   ETANOL 2019        371
     BAYEUX   ETANOL 2020        267
     BAYEUX   ETANOL 2021        310
     BAYEUX   ETANOL 2022        270
     BAYEUX   ETANOL 2023        253
     BAYEUX   ETANOL 2024        290
     BAYEUX   ETANOL 2025        368
     BAYEUX   ETANOL 2026        197
     BAYEUX GASOLINA 2011        430
     BAYEUX GASOLINA 2012        416
     BAYEUX GASOLINA 2013        457
     BAYEUX GASOLINA 2014        412
     BAYEUX GASOLINA 2015        366
     BAYEUX GASOLINA 2016        362
     BAYEUX GASOLINA 2017        287
     BAYEUX GASOLINA 2018        357
     BAYEUX GASOLINA 2019        371
     BAYEUX GASOLINA 2020        280
 

In [11]:
# Conta postos distintos por município, produto e ano em cada arquivo
# como indicador de cobertura, sem somar semestres que podem repetir postos.

resumo_postos = []
for arquivo in arquivos_csv:
    df_temp = ler_csv_anp(
        arquivo,
        sep=";",
        usecols=["Estado - Sigla", "Municipio", "Produto", "Data da Coleta", "CNPJ da Revenda"],
        dtype={"CNPJ da Revenda": "string"}
    )
    df_temp = df_temp[df_temp["Estado - Sigla"] == "PB"].copy()
    df_temp["Municipio"] = df_temp["Municipio"].astype(str).str.strip().str.upper()
    df_temp["Produto"] = df_temp["Produto"].astype(str).str.strip().str.upper()
    df_temp = df_temp[
        df_temp["Municipio"].isin(MUNICIPIOS_ESTUDO)
        & df_temp["Produto"].isin(["GASOLINA", "ETANOL"])
    ].copy()
    df_temp["Data da Coleta"] = pd.to_datetime(
        df_temp["Data da Coleta"], format="%d/%m/%Y", errors="coerce"
    )
    df_temp["Ano"] = df_temp["Data da Coleta"].dt.year
    resumo_postos.append(
        df_temp.groupby(["Municipio", "Produto", "Ano"])["CNPJ da Revenda"]
        .nunique().reset_index(name="Postos")
    )

postos = pd.concat(resumo_postos, ignore_index=True)
postos = postos.groupby(
    ["Municipio", "Produto", "Ano"], as_index=False
)["Postos"].max()
print(postos.to_string(index=False))

C:\Users\super\AppData\Local\Temp\ipykernel_6688\3696483139.py:7: DtypeWarning: Columns (1: Estado - Sigla, 2: Municipio, 10: Produto, 11: Data da Coleta) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(arquivo, encoding="utf-8-sig", **kwargs)


  Municipio  Produto  Ano  Postos
     BAYEUX   ETANOL 2011       9
     BAYEUX   ETANOL 2012       8
     BAYEUX   ETANOL 2013       9
     BAYEUX   ETANOL 2014       8
     BAYEUX   ETANOL 2015      11
     BAYEUX   ETANOL 2016       9
     BAYEUX   ETANOL 2017       9
     BAYEUX   ETANOL 2018       9
     BAYEUX   ETANOL 2019      11
     BAYEUX   ETANOL 2020       9
     BAYEUX   ETANOL 2021       7
     BAYEUX   ETANOL 2022       7
     BAYEUX   ETANOL 2023       6
     BAYEUX   ETANOL 2024       9
     BAYEUX   ETANOL 2025       8
     BAYEUX   ETANOL 2026       9
     BAYEUX GASOLINA 2011       9
     BAYEUX GASOLINA 2012       8
     BAYEUX GASOLINA 2013      10
     BAYEUX GASOLINA 2014       9
     BAYEUX GASOLINA 2015      12
     BAYEUX GASOLINA 2016      11
     BAYEUX GASOLINA 2017      11
     BAYEUX GASOLINA 2018      11
     BAYEUX GASOLINA 2019      11
     BAYEUX GASOLINA 2020       9
     BAYEUX GASOLINA 2021       8
     BAYEUX GASOLINA 2022       8
     BAYEUX GA

In [12]:
# Recalcula a quantidade de postos realmente distintos por município, produto e ano
# juntando os semestres e arquivos antes da contagem.

dados_postos = []
for arquivo in arquivos_csv:
    df_temp = ler_csv_anp(
        arquivo,
        sep=";",
        usecols=["Estado - Sigla", "Municipio", "Produto", "Data da Coleta", "CNPJ da Revenda"],
        dtype={"CNPJ da Revenda": "string"}
    )
    df_temp = df_temp[df_temp["Estado - Sigla"] == "PB"].copy()
    df_temp["Municipio"] = df_temp["Municipio"].astype(str).str.strip().str.upper()
    df_temp["Produto"] = df_temp["Produto"].astype(str).str.strip().str.upper()
    df_temp = df_temp[
        df_temp["Municipio"].isin(MUNICIPIOS_ESTUDO)
        & df_temp["Produto"].isin(["GASOLINA", "ETANOL"])
    ].copy()
    df_temp["Data da Coleta"] = pd.to_datetime(
        df_temp["Data da Coleta"], format="%d/%m/%Y", errors="coerce"
    )
    df_temp["Ano"] = df_temp["Data da Coleta"].dt.year
    dados_postos.append(df_temp)

df_postos = pd.concat(dados_postos, ignore_index=True)
postos_anuais = (
    df_postos.groupby(["Municipio", "Produto", "Ano"])["CNPJ da Revenda"]
    .nunique().reset_index(name="Postos")
)
print(postos_anuais.to_string(index=False))

C:\Users\super\AppData\Local\Temp\ipykernel_6688\3696483139.py:7: DtypeWarning: Columns (1: Estado - Sigla, 2: Municipio, 10: Produto, 11: Data da Coleta) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(arquivo, encoding="utf-8-sig", **kwargs)


  Municipio  Produto  Ano  Postos
     BAYEUX   ETANOL 2011       9
     BAYEUX   ETANOL 2012       8
     BAYEUX   ETANOL 2013       9
     BAYEUX   ETANOL 2014       8
     BAYEUX   ETANOL 2015      12
     BAYEUX   ETANOL 2016      10
     BAYEUX   ETANOL 2017       9
     BAYEUX   ETANOL 2018      10
     BAYEUX   ETANOL 2019      11
     BAYEUX   ETANOL 2020      11
     BAYEUX   ETANOL 2021       8
     BAYEUX   ETANOL 2022       7
     BAYEUX   ETANOL 2023       6
     BAYEUX   ETANOL 2024       9
     BAYEUX   ETANOL 2025      16
     BAYEUX   ETANOL 2026       9
     BAYEUX GASOLINA 2011       9
     BAYEUX GASOLINA 2012       8
     BAYEUX GASOLINA 2013      10
     BAYEUX GASOLINA 2014       9
     BAYEUX GASOLINA 2015      13
     BAYEUX GASOLINA 2016      12
     BAYEUX GASOLINA 2017      11
     BAYEUX GASOLINA 2018      11
     BAYEUX GASOLINA 2019      11
     BAYEUX GASOLINA 2020      11
     BAYEUX GASOLINA 2021       8
     BAYEUX GASOLINA 2022       8
     BAYEUX GA

In [13]:
# Consolida o recorte do estudo e preserva identificadores e endereço dos postos.

COLUNAS_ESTUDO = [
    "Estado - Sigla", "Municipio", "Revenda", "CNPJ da Revenda",
    "Nome da Rua", "Numero Rua", "Complemento", "Bairro", "Cep",
    "Produto", "Data da Coleta", "Valor de Venda", "Bandeira"
]

dados_estudo = []
for arquivo in arquivos_csv:
    df_temp = ler_csv_anp(
        arquivo,
        sep=";",
        usecols=COLUNAS_ESTUDO,
        dtype={"CNPJ da Revenda": "string", "Cep": "string"},
        low_memory=False
    )
    df_temp = df_temp[
        (df_temp["Estado - Sigla"].astype("string").str.strip().str.upper() == "PB")
        & df_temp["Municipio"].astype("string").str.strip().str.upper().isin(MUNICIPIOS_ESTUDO)
        & df_temp["Produto"].astype("string").str.strip().str.upper().isin(["GASOLINA", "ETANOL"])
    ].copy()
    df_temp["Arquivo Origem"] = arquivo.name
    dados_estudo.append(df_temp)

df_estudo = pd.concat(dados_estudo, ignore_index=True)
print("Arquivos processados:", len(arquivos_csv))
print("Registros no recorte:", len(df_estudo))
print("Colunas preservadas:", df_estudo.columns.tolist())

Arquivos processados: 31
Registros no recorte: 69506
Colunas preservadas: ['Estado - Sigla', 'Municipio', 'Revenda', 'CNPJ da Revenda', 'Nome da Rua', 'Numero Rua', 'Complemento', 'Bairro', 'Cep', 'Produto', 'Data da Coleta', 'Valor de Venda', 'Bandeira', 'Arquivo Origem']


In [14]:
# Padroniza os nomes das colunas, preservando o CNPJ original e uma versão normalizada.

df_estudo = df_estudo.rename(columns={
    "Estado - Sigla": "estado",
    "Municipio": "municipio",
    "Revenda": "revenda",
    "CNPJ da Revenda": "cnpj_original",
    "Nome da Rua": "logradouro",
    "Numero Rua": "numero",
    "Complemento": "complemento",
    "Bairro": "bairro",
    "Cep": "cep",
    "Produto": "produto",
    "Data da Coleta": "data_original",
    "Valor de Venda": "valor_original",
    "Bandeira": "bandeira",
    "Arquivo Origem": "arquivo_origem"
})

df_estudo["cnpj_revenda"] = (
    df_estudo["cnpj_original"].astype("string").str.replace("[^0-9]", "", regex=True).str.zfill(14)
)
df_estudo["cep"] = df_estudo["cep"].astype("string").str.replace("[^0-9]", "", regex=True)
df_estudo["data"] = pd.to_datetime(
    df_estudo["data_original"], format="%d/%m/%Y", errors="coerce"
)
df_estudo["valor"] = pd.to_numeric(
    df_estudo["valor_original"].astype("string").str.replace(",", ".", regex=False),
    errors="coerce"
)

print("Registros:", len(df_estudo))
print("CNPJs não vazios:", df_estudo["cnpj_revenda"].notna().sum())
print("Datas não interpretadas:", df_estudo["data"].isna().sum())
print("Preços não interpretados:", df_estudo["valor"].isna().sum())
print(df_estudo.dtypes)

Registros: 69506
CNPJs não vazios: 69506
Datas não interpretadas: 0
Preços não interpretados: 0
estado                       str
municipio                    str
revenda                      str
cnpj_original             string
logradouro                   str
numero                       str
complemento                  str
bairro                       str
cep                       string
produto                      str
data_original                str
valor_original               str
bandeira                     str
arquivo_origem               str
cnpj_revenda              string
data              datetime64[us]
valor                    Float64
dtype: object


In [15]:
# Verifica quantos valores nulos existem em cada coluna da base
# antes de qualquer exclusão ou tratamento

nulos = df_estudo.isna().sum()

print("Valores nulos por coluna:")
print(nulos)

Valores nulos por coluna:
estado                0
municipio             0
revenda               0
cnpj_original         0
logradouro            0
numero                0
complemento       36184
bairro                0
cep                   0
produto               0
data_original         0
valor_original        0
bandeira              0
arquivo_origem        0
cnpj_revenda          0
data                  0
valor                 0
dtype: int64


In [16]:
# Verifica se existem registros duplicados para o mesmo posto, produto, data e preço
# usando o CNPJ como identificador principal do estabelecimento

duplicados = df_estudo.duplicated(
    subset=[
        "cnpj_revenda",
        "produto",
        "data",
        "valor"
    ],
    keep=False
)

print("Registros envolvidos em possíveis duplicidades:")
print(duplicados.sum())

if duplicados.any():
    print("\nExemplos de registros duplicados:")
    
    print(
        df_estudo.loc[
            duplicados,
            [
                "municipio",
                "revenda",
                "cnpj_revenda",
                "produto",
                "data",
                "valor",
                "bandeira"
            ]
        ]
        .sort_values(
            ["cnpj_revenda", "produto", "data", "valor"]
        )
        .head(30)
        .to_string(index=False)
    )

Registros envolvidos em possíveis duplicidades:
0


In [17]:
# Verifica se o mesmo CNPJ aparece associado a mais de um nome de revenda
# ou a mais de uma bandeira ao longo do período analisado

consistencia_postos = (
    df_estudo.groupby("cnpj_revenda")
    .agg(
        nomes_revenda=("revenda", "nunique"),
        bandeiras=("bandeira", "nunique"),
        municipio=("municipio", "nunique")
    )
    .reset_index()
)

print("Quantidade total de CNPJs distintos:")
print(len(consistencia_postos))

print("\nCNPJs com mais de um nome de revenda:")
print((consistencia_postos["nomes_revenda"] > 1).sum())

print("\nCNPJs com mais de uma bandeira:")
print((consistencia_postos["bandeiras"] > 1).sum())

print("\nCNPJs associados a mais de um município:")
print((consistencia_postos["municipio"] > 1).sum())

Quantidade total de CNPJs distintos:
205

CNPJs com mais de um nome de revenda:
15

CNPJs com mais de uma bandeira:
98

CNPJs associados a mais de um município:
0


In [18]:
# Mostra exemplos de postos que aparecem com mais de uma bandeira
# ao longo do período, para verificar se as mudanças parecem plausíveis

cnpjs_multibandeira = consistencia_postos.loc[
    consistencia_postos["bandeiras"] > 1,
    "cnpj_revenda"
]

exemplos_bandeira = (
    df_estudo[
        df_estudo["cnpj_revenda"].isin(cnpjs_multibandeira)
    ][
        [
            "cnpj_revenda",
            "revenda",
            "municipio",
            "bandeira",
            "data"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        ["cnpj_revenda", "data"]
    )
)

print(
    exemplos_bandeira.head(40).to_string(index=False)
)

  cnpj_revenda                                         revenda   municipio bandeira       data
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-01-04
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-01-31
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-02-09
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-06-07
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-06-15
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-06-20
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-06-29
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-07-06
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA   BRANCA 2011-07-13
00289351000164 REVENDA DE COMBUSTÍVEIS BEZERRA CAV

In [19]:
# Resume as diferentes bandeiras associadas a cada CNPJ
# para identificar de forma clara quais postos mudaram de bandeira ao longo do tempo

resumo_bandeiras = (
    df_estudo.groupby(
        [
            "cnpj_revenda",
            "revenda",
            "municipio"
        ]
    )["bandeira"]
    .agg(
        quantidade_bandeiras="nunique",
        bandeiras=lambda x: " | ".join(sorted(x.unique()))
    )
    .reset_index()
)

# Mantém apenas os postos que tiveram mais de uma bandeira
postos_multibandeira = resumo_bandeiras[
    resumo_bandeiras["quantidade_bandeiras"] > 1
].copy()

print("Postos que tiveram mais de uma bandeira:")
print(len(postos_multibandeira))

print("\nExemplos:")
print(
    postos_multibandeira.head(20).to_string(index=False)
)

Postos que tiveram mais de uma bandeira:
98

Exemplos:
  cnpj_revenda                                                 revenda   municipio  quantidade_bandeiras                                                            bandeiras
00289351000164         REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA                     4        BRANCA | PETROBRAS DISTRIBUIDORA S.A. | VIBRA | VIBRA ENERGIA
00378736000106           AGUA FRIA REVENDEDOR DE COMBUSTIVEIS LIMITADA JOAO PESSOA                     2                                  ALVO | PETROBRAS DISTRIBUIDORA S.A.
01256702000101 POSTO CAJUEIRO COMERCIO DE DERIVADOS DE PETROLEO EIRELI JOAO PESSOA                     3                       BRANCA | PETROBRAS DISTRIBUIDORA S.A. | RAIZEN
01633732000181               COMERCIAL DE COMBUSTIVEIS ENTRE RIOS LTDA      BAYEUX                     2                                BRANCA | PETROBRAS DISTRIBUIDORA S.A.
01765332000120                   BEZERRA COMERCIO DE COMBUSTIVEIS LTDA JOAO

In [20]:
# Mostra o primeiro e o último registro de cada bandeira para os postos
# que tiveram mudança de bandeira ao longo do período analisado

periodos_bandeira = (
    df_estudo[
        df_estudo["cnpj_revenda"].isin(
            postos_multibandeira["cnpj_revenda"]
        )
    ]
    .groupby([
        "cnpj_revenda",
        "revenda",
        "municipio",
        "bandeira"
    ])
    .agg(
        primeira_coleta=("data", "min"),
        ultima_coleta=("data", "max"),
        registros=("data", "count")
    )
    .reset_index()
    .sort_values([
        "cnpj_revenda",
        "primeira_coleta"
    ])
)

print(
    periodos_bandeira.head(30).to_string(index=False)
)

  cnpj_revenda                                                 revenda   municipio                     bandeira primeira_coleta ultima_coleta  registros
00289351000164         REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA                       BRANCA      2011-01-04    2016-11-16        354
00289351000164         REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA PETROBRAS DISTRIBUIDORA S.A.      2016-11-23    2020-10-19        130
00289351000164         REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA                VIBRA ENERGIA      2022-09-26    2023-07-10         82
00289351000164         REVENDA DE COMBUSTÍVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA                        VIBRA      2023-07-17    2025-06-23        186
00289351000164         REVENDA DE COMBUSTIVEIS BEZERRA CAVALCANTI LTDA JOAO PESSOA                        VIBRA      2025-07-02    2026-06-29        100
00378736000106           AGUA FRIA REVENDEDOR DE COMBUSTIVEIS LIMITADA JOAO PESSOA

In [21]:
# Verifica se o mesmo posto aparece com mais de uma bandeira
# na mesma data de coleta, o que pode indicar inconsistência cadastral
# ou diferenças entre registros de produtos coletados no mesmo dia

conflito_bandeira_data = (
    df_estudo.groupby(
        [
            "cnpj_revenda",
            "revenda",
            "municipio",
            "data"
        ]
    )["bandeira"]
    .nunique()
    .reset_index(name="quantidade_bandeiras")
)

conflitos = conflito_bandeira_data[
    conflito_bandeira_data["quantidade_bandeiras"] > 1
].copy()

print("Datas com mais de uma bandeira para o mesmo posto:")
print(len(conflitos))

if len(conflitos) > 0:
    exemplos_conflitos = (
        df_estudo.merge(
            conflitos[
                [
                    "cnpj_revenda",
                    "data"
                ]
            ],
            on=[
                "cnpj_revenda",
                "data"
            ],
            how="inner"
        )
        [
            [
                "cnpj_revenda",
                "revenda",
                "municipio",
                "produto",
                "data",
                "bandeira",
                "valor"
            ]
        ]
        .sort_values(
            [
                "cnpj_revenda",
                "data",
                "produto"
            ]
        )
    )

    print("\nExemplos:")
    print(
        exemplos_conflitos.head(40).to_string(index=False)
    )

Datas com mais de uma bandeira para o mesmo posto:
0


In [22]:
# Gera um resumo final da etapa de ingestão do novo recorte
# para registrar as principais características da base utilizada no estudo

resumo_ingestao = {
    "arquivos_processados": len(arquivos_csv),
    "registros_recorte": len(df_estudo),
    "municipios": df_estudo["municipio"].nunique(),
    "postos_cnpj": df_estudo["cnpj_revenda"].nunique(),
    "produtos": df_estudo["produto"].nunique(),
    "data_inicial": df_estudo["data"].min(),
    "data_final": df_estudo["data"].max(),
    "duplicados_exatos": int(duplicados.sum()),
    "datas_com_conflito_bandeira": len(conflitos)
}

for chave, valor in resumo_ingestao.items():
    print(f"{chave}: {valor}")

arquivos_processados: 31
registros_recorte: 69506
municipios: 4
postos_cnpj: 205
produtos: 2
data_inicial: 2011-01-04 00:00:00
data_final: 2026-06-29 00:00:00
duplicados_exatos: 0
datas_com_conflito_bandeira: 0


In [23]:
# Salva a base integrada no caminho consumido pelo notebook 02.

PASTA_INGESTAO = RAIZ / "Dados/Limpo"
PASTA_INGESTAO.mkdir(parents=True, exist_ok=True)
arquivo_ingestao = PASTA_INGESTAO / "precos_rmjp_ingestao.parquet"
df_estudo.to_parquet(arquivo_ingestao, index=False)

print("Arquivo salvo em:", arquivo_ingestao)
print("Tamanho:", arquivo_ingestao.stat().st_size, "bytes")

Arquivo salvo em: c:\Users\super\Desktop\Leo\faculdade\Tcc analise combustivel\projeto\TCC_Combustiveis_revisado_20261014\Dados\Limpo\precos_rmjp_ingestao.parquet
Tamanho: 555936 bytes


In [24]:
# Confere se o PyArrow foi instalado corretamente no mesmo ambiente do notebook

import pyarrow

print("PyArrow:", pyarrow.__version__)

PyArrow: 25.0.1
